# 🧪 Laboratorio · El paso hacia adelante de una LSTM, *a mano* · *Cap. 2.2*
### Redes Neuronales — Deep Learning · Maestría en Ciencia de Datos · Universidad Santo Tomás

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JotaMao1985/Deep_Learning_Usta/blob/main/notebooks/08-lab-lstm-forward-paso-a-paso.ipynb)

Este cuaderno es el **complemento ejecutable** del simulador interactivo *El paso hacia adelante de una LSTM* (`Material html/lstm-forward-interactivo.html`). Mientras el simulador deja **ver y tocar** cada compuerta, aquí se **abre la caja negra en código**: se implementa el paso hacia adelante de una celda LSTM **escalar** con las seis ecuaciones a mano, se **reproduce exactamente** el caso canónico y los escenarios del simulador, y se verifica que ese cálculo coincide con el de **PyTorch** (`nn.LSTMCell`).

> **Cómo usarlo:** ejecutar las celdas en orden (en Google Colab: *Entorno de ejecución → Ejecutar todas*). Corre en **CPU en segundos**: es aritmética escalar, sin entrenamiento. Conviene cambiar los pesos y la secuencia y volver a ejecutar para observar el efecto —igual que en el simulador—.

> ⚠️ **Importante:** se usa una LSTM **escalar de juguete** (una sola unidad, pesos fijos elegidos con fines didácticos), no datos ni modelos reales. **No es la solución de ninguna actividad evaluable**; su propósito es entender la mecánica interna de la celda LSTM. Para entrenar una LSTM sobre una serie real, ver el §3 del `03-lab-arquitecturas-modulo-2.ipynb`.

---
**Contenido**
1. Preparación del entorno
2. Las seis ecuaciones, a mano
3. Reproducir el caso canónico del simulador (paridad)
4. Escenario «cambio de contexto» — la compuerta de olvido limpia la cinta
5. Escenario «puente de dependencia larga» — la señal viaja intacta
6. Las compuertas «se encienden» con su valor real
7. Del escalar al vector: verificación contra PyTorch `nn.LSTMCell`
8. ✍️ Reto (opcional)


## 1. Preparación del entorno

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 1 · Preparación del entorno
# Qué hace · Importa librerías, fija la semilla y configura el estilo USTA.
# Por qué  · Una semilla fija hace comparables las ejecuciones. Aquí no hay azar en el cálculo
#           (es determinista), pero se mantiene la convención del curso.

# En Colab estas librerías ya están instaladas. Si faltara alguna, descomentar:
# !pip -q install numpy matplotlib torch

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

SEMILLA = 42
np.random.seed(SEMILLA)

# Paleta USTA para las gráficas
USTA_MORADO, USTA_ROSA, USTA_NAVY = "#3D008D", "#ED1E79", "#001A4D"
USTA_AMBAR, USTA_VERDE = "#EAB308", "#0D9488"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

print("Entorno listo · numpy", np.__version__)


## 2. Las seis ecuaciones, a mano

En cada paso $t$, la celda recibe la entrada $x_t$, el estado oculto previo $h_{t-1}$ y el estado de celda previo $c_{t-1}$, y calcula:

$$
f_t = \sigma(w_f x_t + u_f h_{t-1} + b_f) \qquad
i_t = \sigma(w_i x_t + u_i h_{t-1} + b_i) \qquad
\tilde{c}_t = \tanh(w_c x_t + u_c h_{t-1} + b_c)
$$
$$
o_t = \sigma(w_o x_t + u_o h_{t-1} + b_o) \qquad
c_t = f_t\,c_{t-1} + i_t\,\tilde{c}_t \qquad
h_t = o_t\,\tanh(c_t)
$$

con $c_0 = h_0 = 0$. Las tres **compuertas** ($f_t, i_t, o_t$) usan la sigmoide $\sigma(z)=\tfrac{1}{1+e^{-z}}\in[0,1]$; la **memoria candidata** $\tilde{c}_t$ usa $\tanh\in[-1,1]$.

La ecuación clave es la de $c_t$: es una **suma** (no una multiplicación encadenada). Esa cinta aditiva —la «cinta transportadora»— es lo que deja fluir el gradiente sin desvanecerse a lo largo de muchos pasos.

> En una LSTM **real**, cada variable es un **vector** y $w_\cdot, u_\cdot$ son **matrices**; las compuertas se aplican elemento a elemento (producto de Hadamard $\odot$). Aquí se usa **dimensión 1** para ver toda la aritmética. La lógica es idéntica.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 2 · El paso hacia adelante de una LSTM escalar (nucleo)
# Qué hace · Implementa las seis ecuaciones y devuelve TODO el historial paso a paso.
# Por qué  · Es exactamente el motor del simulador interactivo, ahora legible en Python.

def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))

def paso_hacia_adelante(entradas, W):
    """LSTM escalar. W: dict con w_/u_/b_ para f,i,c,o. Devuelve lista de dicts por paso."""
    c, h = 0.0, 0.0                       # c_0 = h_0 = 0 (sin contexto previo)
    historial = []
    for t, x in enumerate(entradas):
        f    = sigmoide(W['wf']*x + W['uf']*h + W['bf'])   # compuerta de olvido   [0,1]
        i    = sigmoide(W['wi']*x + W['ui']*h + W['bi'])   # compuerta de entrada  [0,1]
        cand = np.tanh(  W['wc']*x + W['uc']*h + W['bc'])   # memoria candidata     [-1,1]
        o    = sigmoide(W['wo']*x + W['uo']*h + W['bo'])   # compuerta de salida   [0,1]
        c    = f*c + i*cand                                # actualización de la cinta (suma)
        h    = o*np.tanh(c)                                # estado oculto (salida del paso)
        historial.append(dict(t=t, x=float(x), f=f, i=i, cand=cand, o=o, c=c, h=h))
    return historial

# Prueba rápida con un solo paso
demo = paso_hacia_adelante([0.5], dict(wf=0.7,uf=0.1,bf=0.0, wi=0.9,ui=0.2,bi=-0.1,
                                       wc=0.8,uc=0.15,bc=0.05, wo=0.6,uo=0.25,bo=0.0))
print("t=0 → f=%.4f  i=%.4f  cand=%.4f  o=%.4f  c=%.4f  h=%.4f" %
      (demo[0]['f'], demo[0]['i'], demo[0]['cand'], demo[0]['o'], demo[0]['c'], demo[0]['h']))


## 3. Reproducir el caso canónico del simulador (paridad)

El simulador trae un **auto-test** (`?selftest=1`) que compara su motor con una referencia en Python (NumPy). Aquí se hace lo contrario: se corre el motor de este cuaderno con **los mismos pesos y la misma secuencia** y se comprueba que reproduce los **valores dorados** hasta $\varepsilon = 10^{-9}$. Si esta celda pasa, el código de arriba calcula *exactamente* lo que muestra la herramienta.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 3 · Paridad con el simulador (caso canonico)
# Qué hace · Corre el forward pass con los pesos por defecto del simulador y compara con
#           los valores dorados (los mismos que verifica ?selftest=1).
# Por qué  · Garantiza que este cuaderno y la herramienta calculan lo mismo.

W_CANONICO = dict(wf=0.7, uf=0.10, bf=0.00,  wi=0.9, ui=0.20, bi=-0.10,
                  wc=0.8, uc=0.15, bc=0.05,  wo=0.6, uo=0.25, bo=0.00)
X_CANONICO = [0.5, -1.0, 2.0]

# Valores dorados (NumPy, precision completa) incrustados en el simulador:
DORADOS = [
    dict(f=0.5866175789173301, i=0.5866175789173301, cand=0.42189900525000795,
         o=0.574442516811659,  c=0.2474933730073896, h=0.13933732449622496),
    dict(f=0.3349087221738316, i=0.2744556468534132, cand=-0.6225140635013252,
         o=0.36235306816108587,c=-0.08796481067320665,h=-0.03179236024811944),
    dict(f=0.8016789070985031, i=0.8447024561517917, cand=0.9282003159129304,
         o=0.767107845810217,  c=0.7135335533688983, h=0.47015107933959094),
]

hist = paso_hacia_adelante(X_CANONICO, W_CANONICO)

print("paso |   x    |   f     |   i     |  cand   |   o     |   c     |   h")
print("-"*72)
for s in hist:
    print(" %2d  | %+.2f | %.5f | %.5f | %+.5f | %.5f | %+.5f | %+.5f" %
          (s['t'], s['x'], s['f'], s['i'], s['cand'], s['o'], s['c'], s['h']))

eps = 1e-9
todo_ok = all(abs(hist[t][k]-DORADOS[t][k]) <= eps
              for t in range(3) for k in ['f','i','cand','o','c','h'])
print("\n✅ Paridad con el simulador (ε=1e-9):", "OK" if todo_ok else "FALLA")
assert todo_ok, "El forward pass no coincide con los valores dorados del simulador"


## 4. Escenario «cambio de contexto» — la compuerta de olvido limpia la cinta

Mismos pesos y secuencia que el escenario guiado del simulador: entradas positivas sostenidas y luego una entrada **negativa fuerte** ($x=-4$). Se observa cómo la memoria se acumula y cómo, al llegar la entrada negativa, la compuerta de olvido cae a $\approx 0$ y **borra** el estado de celda acumulado.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 4 · Escenario: cambio de contexto
# Qué hace · Corre el escenario y grafica la compuerta de olvido y la cinta c_t.
# Por qué  · Hace visible cómo la LSTM «olvida» información obsoleta ante un giro brusco.

W_CTX = dict(wf=2.0, uf=0.0, bf=1.5,  wi=1.5, ui=0.0, bi=0.0,
             wc=1.0, uc=0.0, bc=0.0,  wo=1.0, uo=0.0, bo=0.5)
X_CTX = [1.0, 1.0, 1.0, -4.0, 1.0]
h = paso_hacia_adelante(X_CTX, W_CTX)
ts = [s['t'] for s in h]; fs = [s['f'] for s in h]; cs = [s['c'] for s in h]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
colores = [USTA_ROSA if x < 0 else USTA_AMBAR for x in X_CTX]
ax[0].bar(ts, fs, color=colores)
ax[0].set_ylim(0, 1); ax[0].set_xlabel("paso t"); ax[0].set_ylabel("f_t")
ax[0].set_title("Compuerta de olvido f_t\n(se cierra en t=3, x=-4)")
for t, v in zip(ts, fs): ax[0].text(t, v+0.02, f"{v:.2f}", ha="center", fontsize=9)

ax[1].plot(ts, cs, "-o", color=USTA_MORADO, lw=2)
ax[1].axhline(0, color="gray", lw=0.8)
ax[1].set_xlabel("paso t"); ax[1].set_ylabel("c_t")
ax[1].set_title("Estado de celda c_t (la cinta)\nse acumula y luego se «limpia»")
for t, v in zip(ts, cs): ax[1].text(t, v+0.05, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

print("En t=2 la memoria vale c=%.3f; en t=3 el olvido cae a f=%.4f y la cinta queda en c=%.4f."
      % (cs[2], fs[3], cs[3]))


## 5. Escenario «puente de dependencia larga» — la señal viaja intacta

Información importante en $t=0$ y luego solo **ruido** ($x\approx 0$). Con la compuerta de olvido anclada en $\approx 1$, la señal inicial recorre la cinta **casi sin degradarse**: así resuelve la LSTM el desvanecimiento del gradiente.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 5 · Escenario: puente de dependencia larga
# Qué hace · Corre el escenario y grafica cómo la señal de t=0 sobrevive el ruido.
# Por qué  · Muestra el mecanismo que preserva memoria a largo plazo.

W_BRI = dict(wf=0.0, uf=0.0, bf=4.0,  wi=5.0, ui=0.0, bi=-1.0,
             wc=2.0, uc=0.0, bc=0.0,  wo=0.0, uo=0.0, bo=2.0)
X_BRI = [2.0, 0.0, 0.0, 0.0, 0.0]
h = paso_hacia_adelante(X_BRI, W_BRI)
ts = [s['t'] for s in h]; fs = [s['f'] for s in h]; cs = [s['c'] for s in h]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(ts, fs, color=USTA_AMBAR); ax[0].set_ylim(0, 1.05)
ax[0].set_xlabel("paso t"); ax[0].set_ylabel("f_t")
ax[0].set_title("Compuerta de olvido f_t ≈ 1\n(la cinta conserva)")
for t, v in zip(ts, fs): ax[0].text(t, v+0.01, f"{v:.3f}", ha="center", fontsize=9)

ax[1].plot(ts, cs, "-o", color=USTA_VERDE, lw=2)
ax[1].set_ylim(0, max(cs)*1.15); ax[1].set_xlabel("paso t"); ax[1].set_ylabel("c_t")
ax[1].set_title("La señal de t=0 viaja por la cinta\ncasi sin degradarse")
for t, v in zip(ts, cs): ax[1].text(t, v+0.02, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

print("Retención c[4]/c[0] = %.3f (%.0f%% de la señal inicial sobrevive tras 4 pasos de ruido)."
      % (cs[-1]/cs[0], 100*cs[-1]/cs[0]))


## 6. Las compuertas «se encienden» con su valor real

En el simulador, el grosor y la opacidad de cada trazo crecen con el valor de su compuerta. Aquí se traduce esa idea a un **mapa de calor**: cada fila es una compuerta y cada columna un paso; el color es la intensidad de la activación. Se ve de un vistazo qué decide la celda en cada instante.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 6 · Mapa de calor de las activaciones
# Qué hace · Dibuja f, i, |candidato| y o a lo largo del escenario de cambio de contexto.
# Por qué  · Replica en estático la metáfora «la compuerta se enciende» del simulador.

h = paso_hacia_adelante(X_CTX, W_CTX)
M = np.array([[s['f'] for s in h],
              [s['i'] for s in h],
              [abs(s['cand']) for s in h],
              [s['o'] for s in h]])

fig, ax = plt.subplots(figsize=(8, 3.2))
im = ax.imshow(M, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(4)); ax.set_yticklabels(["olvido  f", "entrada  i", "|candidato|", "salida  o"])
ax.set_xticks(range(len(h)))
ax.set_xticklabels(["t=%d\nx=%+.0f" % (s['t'], s['x']) for s in h])
for (r, cc), v in np.ndenumerate(M):
    ax.text(cc, r, "%.2f" % v, ha="center", va="center",
            color="white" if v < 0.6 else "black", fontsize=9)
fig.colorbar(im, ax=ax, label="activación (0..1)")
ax.set_title("Las compuertas se «encienden» con su valor real")
plt.tight_layout(); plt.show()


## 7. Del escalar al vector: verificación contra PyTorch `nn.LSTMCell`

¿Es este cálculo «de juguete» el mismo que el de una LSTM real? Sí. Se configura una `nn.LSTMCell` de PyTorch con **tamaño oculto 1** y **los mismos pesos**, y se comprueba que produce idénticos $c_t$ y $h_t$.

Dos detalles al trasladar los pesos:
- PyTorch **apila las cuatro compuertas** en el orden $i, f, g, o$ (entrada, olvido, candidato $g=\tilde{c}$, salida) en `weight_ih`, `weight_hh` y los sesgos.
- PyTorch usa **dos sesgos** (`bias_ih` + `bias_hh`); como nuestro modelo tiene uno solo por compuerta, se pone `bias_hh = 0`.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 7 · Paridad escalar <-> PyTorch nn.LSTMCell (dim=1)
# Qué hace · Copia los pesos al orden i,f,g,o de PyTorch y compara paso a paso.
# Por qué  · Demuestra que la versión a mano ES la misma celda que usa la librería real.

try:
    import torch
    import torch.nn as nn
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print("PyTorch no está disponible aquí; esta celda se omite (en Colab sí corre).", e)

if TORCH_OK:
    torch.manual_seed(SEMILLA)
    W = W_CANONICO
    celda = nn.LSTMCell(input_size=1, hidden_size=1)
    with torch.no_grad():
        # Orden de compuertas en PyTorch: i, f, g(=candidato), o
        celda.weight_ih.copy_(torch.tensor([[W['wi']], [W['wf']], [W['wc']], [W['wo']]]))
        celda.weight_hh.copy_(torch.tensor([[W['ui']], [W['uf']], [W['uc']], [W['uo']]]))
        celda.bias_ih.copy_(torch.tensor([W['bi'], W['bf'], W['bc'], W['bo']]))
        celda.bias_hh.zero_()          # nuestro modelo tiene un solo sesgo por compuerta

    hx = torch.zeros(1, 1); cx = torch.zeros(1, 1)
    c_torch, h_torch = [], []
    for x in X_CANONICO:
        hx, cx = celda(torch.tensor([[float(x)]]), (hx, cx))
        c_torch.append(cx.item()); h_torch.append(hx.item())

    hist = paso_hacia_adelante(X_CANONICO, W_CANONICO)
    c_mano = [s['c'] for s in hist]; h_mano = [s['h'] for s in hist]

    print("paso |   c (a mano)  |  c (PyTorch)  |   h (a mano)  |  h (PyTorch)")
    print("-"*66)
    for k in range(len(X_CANONICO)):
        print(" %2d  | %+.8f | %+.8f | %+.8f | %+.8f" %
              (k, c_mano[k], c_torch[k], h_mano[k], h_torch[k]))

    ok = np.allclose(c_mano, c_torch, atol=1e-5) and np.allclose(h_mano, h_torch, atol=1e-5)
    print("\n✅ Paridad escalar ↔ PyTorch nn.LSTMCell:", "OK" if ok else "FALLA")
    assert ok, "El forward pass a mano no coincide con nn.LSTMCell"


## 8. ✍️ Reto (opcional)

Un buen ejercicio para fijar la intuición de las compuertas: **encontrar pesos** que hagan que la compuerta de olvido permanezca **abierta** ($f_t \gtrsim 0.9$) en los primeros pasos y se **cierre de golpe** ($f_t < 0.1$) solo en el último, para la secuencia $X=[1,1,1,1,5]$.

Pista: con $u_f=0$, se tiene $f_t = \sigma(w_f\,x_t + b_f)$. Hay que elegir el **signo** de $w_f$ y el valor de $b_f$ para que $w_f\,x_t + b_f$ sea grande y positivo cuando $x=1$, y muy negativo cuando $x=5$.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte · ✍️ Reto (opcional)
# Completar los valores marcados con TODO y descomentar la comprobación.

X_RETO = [1, 1, 1, 1, 5]
W_RETO = dict(W_CANONICO)     # se parte de los pesos por defecto
W_RETO['uf'] = 0.0
W_RETO['wf'] = None   # TODO: ¿qué signo debe tener para que x=5 cierre la compuerta?
W_RETO['bf'] = None   # TODO: sesgo que mantenga abierta la compuerta cuando x=1

# --- Comprobación (descomentar cuando W_RETO['wf'] y ['bf'] tengan valores) ---
# h = paso_hacia_adelante(X_RETO, W_RETO)
# fs = [s['f'] for s in h]
# print("f_t por paso:", [f"{v:.3f}" for v in fs])
# assert all(v > 0.9 for v in fs[:-1]) and fs[-1] < 0.1, "Aún no; ajusta wf y bf."
# print("✅ ¡Reto logrado! La compuerta de olvido se cierra solo en el último paso.")


## Puente al material del módulo

Este cuaderno abre la celda LSTM «por dentro»; el `03-lab-arquitecturas-modulo-2.ipynb` (§3) la usa «por fuera» como `nn.LSTM` para **pronosticar una serie temporal**. Juntos cubren las dos caras: la **mecánica interna** (aquí) y el **uso práctico** (allá), que es el que exige la **Rama B (secuencias)** de la Actividad *Centinela · Fase 2*.

### 🔗 Relacionado
- Simulador interactivo del que este cuaderno es complemento: [El paso hacia adelante de una LSTM](https://jotamao1985.github.io/Deep_Learning_Usta/lstm-forward-interactivo.html) (embebido en el [Material del Módulo 2](https://jotamao1985.github.io/Deep_Learning_Usta/02_Modulo_2_Arquitecturas_Deep_Learning.html), Cap. 2.2).
- Olah, C. (2015). [*Understanding LSTM Networks*](http://colah.github.io/posts/2015-08-Understanding-LSTMs/) — origen de la analogía de la «cinta transportadora».
- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*, Cap. 10 (redes recurrentes). MIT Press.
- [PyTorch · `torch.nn.LSTMCell`](https://pytorch.org/docs/stable/generated/torch.nn.LSTMCell.html) — documentación oficial (orden de compuertas y forma de los pesos).
